# 적용하기 — 보도자료 → 기사 첫 문장(리드) 학습 데이터
1. 수집 `press_raw.csv` → 2. 정제 `press_clean.csv` → 3. 리드 초안 `pairs_draft.csv` → 4. `lead_draft.jsonl`

⚠️ 수집 전: robots.txt · 이용약관 · 공공누리 표시 확인. CSV·JSONL은 GitHub에 올리지 않아요.

In [1]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

실행 위치: 내 컴퓨터 (Windows)
GPU: NVIDIA GeForce RTX 5070 Laptop GPU, 8151 MiB
한글 글꼴: ['Malgun Gothic']


In [2]:
# ▶ API 키 불러오기 — Colab은 🔑 보안 비밀, VS Code는 같은 폴더의 .env 파일
#   .env 파일 내용 예:  OPENAI_API_KEY=sk-...   (GitHub에 올리지 않기)
def get_key(name='OPENAI_API_KEY'):
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        print('pip install python-dotenv 후 다시 실행')
    return os.getenv(name)

OPENAI_API_KEY = get_key()
print('API 키:', '준비됨' if OPENAI_API_KEY else '없음 → 키를 등록하세요')

API 키: 없음 → 키를 등록하세요


## 1. 수집 — 주황 칸 4개를 F12로 찾아 바꾸기

In [ ]:
import requests, time, pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# LIST_URL = '목록 주소 (페이지 번호 자리는 {})'
# LINK, TITLE, BODY = '목록 링크 선택자', '제목 선택자', '본문 선택자'
LIST_URL = 'https://www.mcst.go.kr/site/s_notice/press/pressList.jsp?pCurrentPage={}'
LINK = "a[href*='pressView.jsp']"     # href에 'pressView.jsp'가 들어간 링크 = 기사 링크
H = {'User-Agent': 'Mozilla/5.0'}
rows = []
for page in range(1, 2):              # 먼저 1페이지만
    soup = BeautifulSoup(requests.get(LIST_URL.format(page), headers=H).text, 'html.parser')
    for a in soup.select(LINK):
        url = urljoin(LIST_URL, a['href'])
        title = a.get_text(strip=True).replace('새글', '').strip()   # 제목은 목록에서 바로
        rows.append({'제목': title, '링크': url})
        time.sleep(1)
print(len(rows), '건'); rows[:3]

# d = BeautifulSoup(requests.get(rows[0]['링크'], headers=H).text, 'html.parser')
# for box in d.find_all(['div', 'td'], class_=True):
#     text = box.get_text(' ', strip=True)
#     if 300 < len(text) < 8000:
#         print(box.name, box.get('class'), len(text), '|', text[:40])
d = BeautifulSoup(requests.get(rows[0]['링크'], headers=H).text, 'html.parser')
print(rows[0]['제목'])
for box in d.find_all(['div', 'td', 'section', 'article']):
    if not (box.get('class') or box.get('id')):
        continue
    text = box.get_text(' ', strip=True)
    if 200 < len(text) < 8000:
        print(box.name, 'class=', box.get('class'), 'id=', box.get('id'), len(text), '|', text[:40])
# for page in range(1, 11):      # 먼저 range(1, 2)로 한 페이지만 시험
#     soup = BeautifulSoup(requests.get(LIST_URL.format(page), headers=H).text, 'html.parser')
#     for a in soup.select(LINK):
#         d = BeautifulSoup(requests.get(urljoin(LIST_URL, a['href']), headers=H).text, 'html.parser')
#         rows.append({'제목': d.select_one(TITLE).get_text(strip=True),
#                      '본문': d.select_one(BODY).get_text(' ', strip=True)})
#         time.sleep(1)
# pd.DataFrame(rows).to_csv('press_raw.csv', index=False, encoding='utf-8-sig')
# print(len(rows), '건 수집')

10 건
창립 20주년 한국도핑방지위원회, 새로운 도약 다짐
div class= None id= wrap 4268 | 본문 바로가기 주메뉴 바로가기 이 누리집은 대한민국 공식 전자정부 누리집
div class= ['header-in'] id= None 1417 | 어린이 누리집 언어변경 현재 언어 한국어 English (영어) 中文 (
div class= ['inner'] id= None 1301 | 정보공개 정보공개안내 정보공개안내 정보공개제도 소개 비공개 세부기준 정보
div class= ['gnb-toggle-wrap'] id= None 319 | 정상화 과제 제안 창구 모두의 문화예산 국민의견 수렴 청년소리의 정원 청
div class= ['gnb-main-list'] id= None 319 | 정상화 과제 제안 창구 모두의 문화예산 국민의견 수렴 청년소리의 정원 청
div class= ['krds-main-menu-mobile'] id= mobile-nav 1246 | 어린이 누리집 언어변경 현재 언어 한국어 English (영어) 中文 (
div class= ['gnb-wrap'] id= None 1246 | 어린이 누리집 언어변경 현재 언어 한국어 English (영어) 中文 (
div class= ['gnb-body'] id= None 1173 | 정보공개 국민참여 알림·소식 문화광장 국제교류 자료공간 주요정책 문체부소
div class= ['gnb-menu'] id= None 1173 | 정보공개 국민참여 알림·소식 문화광장 국제교류 자료공간 주요정책 문체부소
div class= ['submenu-wrap'] id= None 1126 | 정보공개 정보공개안내 정보공개제도 소개 비공개 세부기준 정보목록 정보공개
div class= ['gnb-sub-list'] id= mGnb-anchor2 284 | 국민참여 정상화 과제 제안 창구 모두의 문화예산 국민의견 수렴 청년소리의
div class= None id= container 1185 | 알림·소

In [12]:
BODY = 'div.여기에_찾은_이름'
for r in rows:
    d = BeautifulSoup(requests.get(r['링크'], headers=H).text, 'html.parser')
    box = d.select_one(BODY)
    if box is None:                      # 못 찾으면 비워 두고 다음으로
        print('본문 못 찾음:', r['링크'])
        r['본문'] = None
    else:
        r['본문'] = box.get_text(' ', strip=True)
    time.sleep(1)
df = pd.DataFrame(rows)
print('본문 있는 글:', df['본문'].notna().sum(), '/', len(df))
df.to_csv('press_raw.csv', index=False, encoding='utf-8-sig')

본문 못 찾음: https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22838
본문 못 찾음: https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22837
본문 못 찾음: https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22836
본문 못 찾음: https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22835
본문 못 찾음: https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22834
본문 못 찾음: https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22833
본문 못 찾음: https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22832
본문 못 찾음: https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22831
본문 못 찾음: https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22830
본문 못 찾음: https://www.mcst.go.kr/site/s_notice/press/pressView.jsp?pSeq=22829
본문 있는 글: 0 / 10


In [20]:
import re, html
VIEW = 'https://www.mcst.go.kr/attachFiles/viewer/result/press/'

def page_text(xml):
    lines, cur, prev_t = [], '', None
    for attrs, ch in re.findall(r"<text\b([^>]*)>(.*?)</text>", xml, re.S):   # 태그 모양을 넓게
        m = re.search(r"\bt\s*=\s*['\"]([-\d.]+)", attrs)                   # 작은/큰따옴표 모두
        t = float(m.group(1)) if m else prev_t
        if prev_t is not None and t is not None and abs(t - prev_t) > 5:
            lines.append(cur.strip()); cur = ''
        cur += html.unescape(ch); prev_t = t
    lines.append(cur.strip())
    return '\n'.join(l for l in lines if l)

def get_xml(url, referer):
    r = requests.get(url, headers={**H, 'Referer': referer})              # 어디서 왔는지 함께 보내기
    r.encoding = 'utf-8'
    return r

def press_body(view_url, max_pages=2, debug=False):
    page = requests.get(view_url, headers=H).text
    m = re.search(r"fn=([^&\"'\s]+?\.pdf)", page)
    if not m:
        return None
    fn = m.group(1)
    doc_url = f'https://www.mcst.go.kr/attachFiles/viewer/skin/doc.html?fn={fn}&rs=/attachFiles/viewer/result/press/'
    idx = get_xml(f'{VIEW}{fn}.xml', doc_url)
    n = int(re.search(r'<pdf_cnt>(\d+)</pdf_cnt>', idx.text).group(1))
    texts = []
    for i in range(1, min(n, max_pages) + 1):
        r = get_xml(f'{VIEW}{fn}_{i}.xml', doc_url)
        if debug:
            print(i, '쪽:', r.status_code, len(r.text)); print(r.text[:400])
        texts.append(page_text(r.text))
        time.sleep(0.5)
    return '\n'.join(texts)

print(press_body(rows[0]['링크'], debug=True)[:600])

1 쪽: 404 2258


<!doctype html>



















<html lang="ko">
<head>
  <meta charset="UTF-8">
  <meta http-equiv="X-UA-Compatible" content="IE=edge" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, minimum-scale=1.0, user-scalable=no" />
  <meta name="keywords" content="" />
  <meta name="description" content="" />
  <meta name="fo
2 쪽: 404 2258


<!doctype html>



















<html lang="ko">
<head>
  <meta charset="UTF-8">
  <meta http-equiv="X-UA-Compatible" content="IE=edge" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, minimum-scale=1.0, user-scalable=no" />
  <meta name="keywords" content="" />
  <meta name="description" content="" />
  <meta name="fo




In [21]:
for r in rows:                                               # ② 전체
    r['본문'] = press_body(r['링크'])
    if r['본문'] is None:
        print('문서 번호 못 찾음:', r['제목'])
    time.sleep(1)
df = pd.DataFrame(rows)
print('본문 있는 글:', df['본문'].notna().sum(), '/', len(df))
df.to_csv('press_raw.csv', index=False, encoding='utf-8-sig')

본문 있는 글: 10 / 10


## 2. 정제 — 공백 정리 · 빈 본문 · 중복 · 길이(200~2,500자)

In [22]:
df = pd.read_csv('press_raw.csv')
print('처음:', len(df))

df['본문'] = df['본문'].str.replace(r'\s+', ' ', regex=True).str.strip()
df = df.dropna(subset=['본문'])
df = df.drop_duplicates(subset=['본문'])
df = df[df['본문'].str.len().between(200, 2500)]

print('정제 후:', len(df))
df.to_csv('press_clean.csv', index=False, encoding='utf-8-sig')


처음: 10
정제 후: 0


In [26]:
print(press_body(rows[0]['링크'], debug=True)[:600])

1 쪽: 404 2258


<!doctype html>



















<html lang="ko">
<head>
  <meta charset="UTF-8">
  <meta http-equiv="X-UA-Compatible" content="IE=edge" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, minimum-scale=1.0, user-scalable=no" />
  <meta name="keywords" content="" />
  <meta name="description" content="" />
  <meta name="fo
2 쪽: 404 2258


<!doctype html>



















<html lang="ko">
<head>
  <meta charset="UTF-8">
  <meta http-equiv="X-UA-Compatible" content="IE=edge" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, minimum-scale=1.0, user-scalable=no" />
  <meta name="keywords" content="" />
  <meta name="description" content="" />
  <meta name="fo




## 3. 리드 초안 — GPT가 쓰고, 확정은 사람이
키 준비: Colab은 왼쪽 🔑(보안 비밀)에 `OPENAI_API_KEY` 등록 → 노트북 액세스 켜기 / VS Code는 `.env` 파일

In [24]:
from dotenv import load_dotenv, find_dotenv
path = find_dotenv(usecwd=True)                 # 지금 폴더부터 위로 올라가며 .env 찾기
print('.env 위치:', path or '못 찾음')
load_dotenv(path, override=True)                # 예전에 읽은 빈 값 덮어쓰기
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
print('API 키:', (OPENAI_API_KEY[:7] + '…') if OPENAI_API_KEY else '없음')

.env 위치: c:\workspace\ai-datajournalism-labs\.env
API 키: sk-proj…


In [25]:
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

def draft_lead(body):
    prompt = f"""다음 보도자료로 기사 첫 문장(리드)을 한 문장으로 써.
- 보도자료에 없는 숫자·원인·전망은 쓰지 마.
- 누가·언제·무엇을 담아.
보도자료: {body[:2000]}"""
    r = client.chat.completions.create(model='gpt-4.1-mini',
                                       messages=[{'role': 'user', 'content': prompt}])
    return r.choices[0].message.content.strip()

df = pd.read_csv('press_clean.csv')
print(draft_lead(df['본문'].iloc[0]))      # 먼저 1개로 시험

IndexError: single positional indexer is out-of-bounds

In [ ]:
df['리드'] = df['본문'].apply(draft_lead)
df.to_csv('pairs_draft.csv', index=False, encoding='utf-8-sig')

## 4. JSONL로 저장 — 한 줄 = 한 쌍 (alpaca 형식)

In [ ]:
import json
INST = '보도자료를 읽고 기사 첫 문장(리드)을 한 문장으로 쓰세요.'
with open('lead_draft.jsonl', 'w', encoding='utf-8') as f:
    for _, r in df.iterrows():
        row = {'instruction': INST, 'input': r['본문'], 'output': r['리드']}
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print(open('lead_draft.jsonl', encoding='utf-8').readline()[:200])

다음 단계: `pairs_draft.csv`를 엑셀로 열어 **판정(사용·수정·제외)** 칸을 채우고 `pairs_checked.csv`로 저장